In [66]:
print('hello !')

hello !


Let's START !

In [67]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict) :
    messages : Annotated[list, add_messages]
    humanRequired : bool
    category : str

In [68]:
from typing import Literal
from pydantic import BaseModel

#structured output for classifier llm
class classifier(BaseModel) : 
    category : Literal['order', 'refund', 'payment', 'human']

In [69]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

classifierLLM = ChatOpenAI(model='gpt-5-nano')
classifierLLM_with_SO = classifierLLM.with_structured_output(classifier)

In [70]:
#classifier agent
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

def classifierAgent(old_state : State) :
    PROMPT = f''' 
    You are a classifier agent for an ecommerce platform. Your work is to analyze the incoming user queries and decide
    the category which they belong to. Your response should be the category the query belongs to.
    Here's the user query - {old_state['messages'][0].content}

    If it belongs to orders specifically - route it to order agent.
    If it belongs to refunds specifically - route it to refund agent.
    If it belongs to payment specifically - route it to payment agent.
    If the query belongs to anything other than the 3 ccategories defined above or if the query is a customer's complaint,
    regarding any order, route them to the compliant agent to raise a ticket.
    '''
    response = classifierLLM_with_SO.invoke(
        input=[SystemMessage(content=PROMPT)]
    )
    print(f'category : {response.category}')
    return {
        'category' : response.category
    }
    

In [71]:
def classify_route(old_state : State) :
    return 'order' if old_state['category']=='order' else 'refund' if old_state['category']=='refund' else 'payment' if old_state['category'] == 'payment' else 'human'

In [ ]:
# check_refund_eligibility tool
from datetime import datetime
from langchain_core.tools import Tool
import importlib
import sqlite_db
from sqlite_db import fetch_order_from_db
importlib.reload(sqlite_db)

def check_refund_eligibility(order_id) :
    '''use this to check if a order is eligible for refund or not'''
    
    order = fetch_order_from_db(order_id)
    order_date_str = order['order_date']  # Example: "2026-05-29"
    order_date = datetime.strptime(order_date_str, "%Y-%m-%d")
    current_date = datetime.now()
    days_passed = (current_date - order_date).days
    
    if order['refund_eligible'] and days_passed<=20 and order['shipping_status'] != 'cancelled' :
        return 'item is eligible for a refund'
    else:
        reason = order['refund_reason'] if not order['refund_eligible'] else 'More than 10 days since order got placed' if days_passed>10 else order['shipping_status']
        return reason
        
check_refund_eligibility_tool =  Tool(
    name='check_refund_eligibility',
    func=check_refund_eligibility,
    description='use this to check if a order is eligible for refund or not'
)

refund_tools = [check_refund_eligibility_tool]

refund_llm = ChatOpenAI(model='gpt-5-nano')
refund_llm_with_tools = refund_llm.bind_tools(refund_tools)

In [73]:
from langchain_core.tools import Tool

def get_order(order_id) :
    '''Use this to fetch order details from the mock DB'''
    print('Tool Called !')

    orderDetails = fetch_order_from_db(order_id)
    # print(f'found order details - {orderDetails}')
    return orderDetails

orderTool = Tool(
    name='fetch_order_details',
    func=get_order,
    description='Use this to fetch order details from the mock DB'
)

order_related_tools = [orderTool]

worker_llm = ChatOpenAI(model='gpt-5-nano')
worker_llm_w_tools = worker_llm.bind_tools(order_related_tools)


In [74]:
#Args schema for the tool
from pydantic import Field

class CreateTicketInput(BaseModel):
    order_id: str = Field(description="The unique order ID string.")
    description: str = Field(
        description="A concise summary of the complaint."
    )

In [ ]:
#TICKET CREATION TOOL 
import sqlite3
import uuid
from langchain_core.tools import StructuredTool

def create_ticket(order_id : str, description : str) : 
    '''use this to create a new ticket for the user query'''
    print('Starting ticket tool  !')
    order = fetch_order_from_db(order_id)
    if not order:
        return 'Order not found, so no ticket can be created !'
    else:
        ticket_conn = sqlite3.connect('tickets.db')
        tick_cursor = ticket_conn.cursor()
        ticket_id = 'TICKET'+str(uuid.uuid4())

        existing_ticket = tick_cursor.execute(
            ''' 
            select * from tickets where order_id = ?
            ''',(order_id,)
        )
        old_ticket = existing_ticket.fetchone()
        if old_ticket :
            return f'ticket already exists - {old_ticket}'
        
        tick_cursor.execute(
            ''' 
            insert into tickets (
                ticket_id,
                order_id,       
                description,     
                status,          
                created_date 
            )
            VALUES (
                ?,?,?,?,?
            )
            ''',
            (
                ticket_id,
                order_id,
                description,
                'Under review',
                '15th June,2026'
            )
        )
        ticket_conn.commit()
        ticket_conn.close()
    print(f'ticket id created - {ticket_id}')
    return f"Success! Created ticket with ID: {ticket_id}"


def get_ticket_details(ticket_id : str) : 
    '''use this tool to find details of customer's tickets'''
    print('inside fetch tickets details tool')
    ticket_conn = sqlite3.connect('tickets.db')
    tick_cursor = ticket_conn.cursor()
    
    ticket = tick_cursor.execute(
        ''' 
        select * from tickets where ticket_id = ?
        ''' , (ticket_id,)
    )
    if not ticket:
        return 'No ticket found with the given ticket id'
    else:
        return ticket.fetchone()

get_tick_details_tool = Tool(
    name='get_ticket_tool',
    func=get_ticket_details,
    description='use this to fetch ticket details'
)

create_tick_tool = StructuredTool.from_function(
    name='create_ticket_tool',
    func = create_ticket,
    description='use this to create a new ticket for the user complaint',
    args_schema = CreateTicketInput
)

human_tools = [create_tick_tool,get_tick_details_tool]
human_llm = ChatOpenAI(model='gpt-5-nano')
human_llm_with_tools = human_llm.bind_tools(human_tools)
#human_llm_with_tools = human_llm_with_tools.with_structured_output(complaint_SO)

In [76]:
from langchain_core.messages import ToolMessage


def orderAgent(old_state:State) :
    
    original_user_query = old_state['messages'][0].content
    
    PROMPT = f''' 
    You are an Order Support Agent for an e-commerce platform.
    Here is the user's question - \n
    User Query:
    {original_user_query}
    
    And here's the conversation so far - \n
    {old_state['messages']}
    
    Instructions:

    * Use the fetch_order_details tool whenever order information is required.
    * Answer ONLY using information returned by the tool.
    * Do not invent tracking details, delivery dates, statuses, or actions by yourself.
    * If the order is not found, respond with a short humorous message.
    * Keep responses concise and customer-friendly.
    If a ToolMessage is already present in the conversation history,
    use that information to answer the user.

    Do not call fetch_order_details again once order information has been retrieved.
    If a ToolMessage is already present in the conversation history,
    DO NOT call any tool again.

    Use the ToolMessage content to answer the user directly.
    
    When order information is available:

    - First identify what the user is asking.
    - Answer ONLY that question.
    - DO NOT DUMP ALL order fields.
    - Summarize and only give the relevant information in plain ENGLISH in one or two sentences.
    
    Examples :
    query - where is my order
    response - your order has been delivered on 13th June 2026.
    
    query - why my payment got failed?
    response - your payment got failed due to this error code - ABC123. Please check details and try again

    '''
    
    if isinstance(old_state["messages"][-1], ToolMessage):
        orderResponse = worker_llm.invoke(
            input=[SystemMessage(content=PROMPT)]
        )
    else:
        orderResponse = worker_llm_w_tools.invoke(
            input=[SystemMessage(content=PROMPT)]
        )
    
    return {
        'messages' : [orderResponse]
    }

In [77]:
def refund_agent(old_state) :
    original_user_query = old_state['messages'][0].content
    
    PROMPT = f''' 
    You are a Refund Support Agent for an e-commerce platform.

    User Query:
    {original_user_query}
    
    And here's the conversation so far - \n
    {old_state['messages']}

    Instructions:

    * Use the check_refund_eligibility tool to determine refund eligibility.
    * Base your response only on the tool output.
    * If the order is eligible, clearly inform the customer.
    * If the order is not eligible, explain the reason returned by the tool.
    * Do not invent refund policies or business rules.
    * Keep responses concise and professional.
    
    When order information is available:

    - First identify what the user is asking.
    - Answer ONLY that question.
    - Do not dump all order fields.
    - Summarize only the relevant information.

    Examples:

    User: Where is my order?
    Answer: Your order has been delivered on 2026-05-22.

    User: What is my payment status?
    Answer: Your payment was successful.

    User: Is my order delivered?
    Answer: Yes, your order was delivered on 2026-05-22.
    '''
    refund_response = refund_llm_with_tools.invoke(
        input=[SystemMessage(content=PROMPT)]
    )
        
    return {
        'messages' : [refund_response]
    }

In [78]:
def paymentAgent(old_state) :
    original_user_query = old_state['messages'][0].content
    PROMPT = f''' 
    You are a Payment Support Agent for an e-commerce platform.

    User Query:
    {original_user_query}
    
    And here's the conversation so far - \n
    {old_state['messages']}

    Instructions:

    * Use the fetch_order_details tool whenever payment information is needed.
    * Answer only using information returned by the tool.
    * Explain payment status clearly and concisely.
    * Do not invent payment actions, refunds, retries, or support processes.
    * If the order is not found, clearly inform the customer.
    * Keep responses short and professional.
    If a ToolMessage is already present in the conversation history,
    use that information to answer the user.

    Do not call fetch_order_details again once order information has been retrieved.
    If a ToolMessage is already present in the conversation history,
    DO NOT call any tool again.

    Use the ToolMessage content to answer the user directly.
    
    When order information is available:

    - First identify what the user is asking.
    - Answer ONLY that question.
    - DO NOT DUMP ALL order fields.
    - Summarize and only give the relevant information in plain ENGLISH in one or two sentences.

    Examples:

    User: Where is my order?
    Answer: Your order has been delivered on 2026-05-22.

    User: What is my payment status?
    Answer: Your payment was successful.

    User: Is my order delivered?
    Answer: Yes, your order was delivered on 2026-05-22.
    '''
    payment_response = worker_llm_w_tools.invoke(
        input=[SystemMessage(content=PROMPT)]
    )
    
    return {
        'messages' : [payment_response]
    }

In [ ]:
def complaintAgent(old_state : State):
    
    orig_query = old_state['messages'][0].content
    PROMPT = f''' 
    You are a complaint handling agent for a e-commerece business. This is the user's original query :-
    \n {orig_query}
    
    and this is the conversation so far \n 
    {old_state['messages']}
    
    Just like a real complaint handling agent, analyze the query if you can answer the user query directly from
    your intelligence and resolve it. 
    If the user's query is about knowing the details of a ticket id he might have raised, use the get_tick_details_tool
    tool to fetch the ticket details. If you get the details of the ticket back from the tool, analyze it and provide
    the user answer from the fetched ticket details in one line or two.
    
    If the tool returns that there was no ticket found, respond to the user that no such ticket exists with the 
    provided ticket id. Also ask the user to provide the issue so that you can help or create a new ticket.
    
    For anything other than answering a query related to information of a ticket,
    firstly use the create_ticket tool to check if a ticket already exists for the given order id from user. 
    if the tool responds with a existing ticket id, respond to the user saying a ticket already exists for the given 
    order id and provide the ticket details as well. 
    if there is no existing ticket, then use the create_ticket tool to create a support ticket.
    the tool takes order_id and description of issue as parameters and returns back the ticket id
    When the ticket has been created by the tool once, let customer know that a ticket has been created and will be reviewed by out 
    escalation team. Also provide the user with the ticket id for his query.  
    ONCE YOU GOT BACK THE TICKET_ID FROM THE TOOL, DO NOT RUN IT AGAIN.  
    '''
    
    complaint_response = human_llm_with_tools.invoke(
        input=[SystemMessage(content=PROMPT)]
    )
    
    return {
        'messages' : [complaint_response]
    }

In [80]:
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

graph_builder = StateGraph(State)
graph_builder.add_node('classifier', classifierAgent)
graph_builder.add_node('orderAgent', orderAgent)
graph_builder.add_node('fetchOrderTool', ToolNode(tools=order_related_tools))
graph_builder.add_node('refundTools', ToolNode(tools=refund_tools))
graph_builder.add_node('paymentTool', ToolNode(tools=order_related_tools))
graph_builder.add_node('create_ticket_tool', ToolNode(tools=human_tools))
graph_builder.add_node('refundAgent', refund_agent)
graph_builder.add_node('paymentAgent', paymentAgent)
graph_builder.add_node('complaintAgent', complaintAgent)

graph_builder.add_edge(START, 'classifier')
graph_builder.add_conditional_edges('classifier', classify_route,{'order' : 'orderAgent', 'refund' : 'refundAgent', 'payment' : 'paymentAgent', 'human':'complaintAgent'})
graph_builder.add_conditional_edges('orderAgent', tools_condition, {'tools':'fetchOrderTool','__end__' : END})
graph_builder.add_conditional_edges('refundAgent', tools_condition, {'tools':'refundTools','__end__' : END})
graph_builder.add_conditional_edges('paymentAgent', tools_condition, {'tools':'paymentTool','__end__' : END})
graph_builder.add_conditional_edges('complaintAgent', tools_condition, {'tools':'create_ticket_tool','__end__' : END})
graph_builder.add_edge('fetchOrderTool', 'orderAgent')
graph_builder.add_edge('refundTools', 'refundAgent')
graph_builder.add_edge('paymentTool', 'paymentAgent')
graph_builder.add_edge('create_ticket_tool', 'complaintAgent')

graph = graph_builder.compile()


In [ ]:
from IPython.display import display, Image
display(Image(graph.get_graph().draw_mermaid_png()))

In [82]:
import os
print("LangSmith Tracing Enabled:", os.getenv("LANGSMITH_TRACING"))

LangSmith Tracing Enabled: true


In [ ]:
#ORDER QUERY
state = {
    'messages' : [HumanMessage(content='When will ORD1008 arrive?')]
}
res = graph.invoke(state)
print(res['messages'][-1].content)

In [84]:
#REFUND QUERY
state = {
    'messages' : [HumanMessage(content='is my order ORD1005 for a refund?')]
}
res = graph.invoke(state)
print(res['messages'][-1].content)

category : refund
<class 'dict'>
{'order_id': 'ORD1005', 'customer_name': 'Karan Mehta', 'order_date': '2026-05-22', 'amount': 4599.0, 'payment_status': 'Success', 'payment_method': 'Net Banking', 'payment_transaction_id': 'TXN1005', 'payment_error_code': None, 'shipping_status': 'Delivered', 'tracking_number': 'TRK9001005', 'estimated_delivery': '2026-05-29', 'delivery_date': '2026-05-28', 'refund_status': 'Approved', 'refund_reason': 'Wrong item delivered', 'refund_eligible': 1}
ORD1005 is not eligible for a refund. Reason: More than 10 days since the order was placed.


In [ ]:
#PAYMENT QUERY
state = {
    'messages' : [HumanMessage(content='why payment of my order ORD1004 got failed? reason?')]
}
res = graph.invoke(state)
print(res['messages'][-1].content)

In [ ]:
#Complaint Agent
state = {
    'messages' : [HumanMessage(content='I got an wrong item delivered in my order ORD1019 !')]
}
res = graph.invoke(state)
print(res['messages'][-1].content)